In [ ]:
!pip install SPARQLWrapper
!pip install sparql-dataframe

In [ ]:
import sparql_dataframe

endpoint = "http://localhost:7200/repositories/mqtt4ssn"

In [ ]:
NS = {
    "mqtt:": "https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "ex:": "http://example.org/",
    "sosa": "http://www.w3.org/ns/sosa/" 
}

In [ ]:
import pandas as pd

def strip_namespaces(df, ns_map=NS):
    def shrink(val):
        if not isinstance(val, str):
            return val
        # ersetze bekannte Namespaces durch Prefix
        for pref, base in ns_map.items():
            if val.startswith(base):
                return pref + val[len(base):]
        # fallback: nur local name nach # oder /
        if "#" in val:
            return val.rsplit("#", 1)[-1]
        if "/" in val:
            return val.rsplit("/", 1)[-1]
        return val

    return df.map(shrink)


## Control Packet Hierachy

### CQ1.01u - Which MQTT Clients send which MQTT Control Packets?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?client ?packet ?type WHERE {
  ?client a mqtt:Client ;
          mqtt:sendsControlPacket ?packet .

  ?packet a mqtt:ControlPacket ;
          a ?type .

  ?type rdfs:subClassOf* mqtt:ControlPacket .

  FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
  FILTER( ?type != mqtt:ControlPacket )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.02u - Which MQTT Control Packets were sent (specializations)?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?client ?packet ?type WHERE {
  ?client a mqtt:Client ;
          mqtt:sendsControlPacket ?packet .
  ?packet a mqtt:ControlPacket ;
          a ?type .

  ?type rdfs:subClassOf* mqtt:ControlPacket .

  FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
  FILTER( ?type != mqtt:ControlPacket )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.03u - Which MQTT Brokers sends which MQTT Control Packets?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?broker ?packet ?type WHERE {
  ?broker a mqtt:Broker ;
          mqtt:sendsControlPacket ?packet .
  ?packet a mqtt:ControlPacket ;
          a ?type .

  ?type rdfs:subClassOf* mqtt:ControlPacket .

  FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
  FILTER( ?type != mqtt:ControlPacket )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.04u - Which MQTT Control Packets were sent by which Client (inverse)?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?client ?packet ?type WHERE {
  ?client a mqtt:Client ;
          mqtt:sendsControlPacket ?packet .
  ?packet a mqtt:ControlPacket ;
          a ?type .

  ?type rdfs:subClassOf* mqtt:ControlPacket .

  FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
  FILTER( ?type != mqtt:ControlPacket )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.05u - Which MQTT Publish packets carry Observations for a specific SOSA FeatureOfInterest or Property?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/> 

SELECT ?packet ?message ?observationCollection ?foi ?property WHERE {
  ?packet a mqtt:PublishPacket ;
          mqtt:hasPublishPayload ?payload .
  ?payload a mqtt:Payload ;
          mqtt:hasApplicationMessage ?message .
  ?message a mqtt:ApplicationMessage ;       
          mqtt:applicationMessageEncodesObservationCollection ?observationCollection .
  ?activity a mqtt:Activity .
  OPTIONAL { ?activity sosa:hasFeatureOfInterest ?foi }
  OPTIONAL { 
    ?observationCollection sosa:hasMember ?observation .
    ?observation a sosa:Observation ;
                 sosa:observedProperty ?property .
  }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.06u - Which MQTT SUBSCRIBE packets request QoS=2 and for which filters?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?packet ?entry ?filter WHERE {
  ?packet a mqtt:SubscribePacket ;
          mqtt:hasSubscribePayload ?payload .
  ?payload mqtt:hasSubscriptions ?entry .
  ?entry a mqtt:SubscriptionEntry .
  OPTIONAL { ?entry mqtt:hasQoSLevel 2 }
  OPTIONAL { ?entry mqtt:hasTopicFilter ?filter }

}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.07u - Which MQTT UNSUBSCRIBE packets remove all filters under a certain path (e.g. 'factory/centering/#')?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?packet ?filter WHERE {
  ?packet a mqtt:UnsubscribePacket ;
          mqtt:hasUnsubscribePayload ?payload .
  ?payload mqtt:hasTopicFilter ?filter .
  FILTER(STRSTARTS(?filter, "factory/centering/"))
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.08u - Which MQTT Control Packet has which Fixed Header / Variable Header / Payload?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?packet ?type ?fixed ?variable ?payload WHERE {
  ?packet a mqtt:ControlPacket ;
       a ?type.

  OPTIONAL { ?packet mqtt:hasFixedHeader ?fixed }
  OPTIONAL { ?packet mqtt:hasVariableHeader ?variable }
  OPTIONAL { ?packet mqtt:hasPayload ?payload }


  ?type rdfs:subClassOf* mqtt:ControlPacket .

  FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
  FILTER( ?type != mqtt:ControlPacket )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.09u - Which MQTT PUBLISH packet has which MQTT Topic (name) in the Variable Header?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?packet ?topic WHERE {
  ?packet a mqtt:PublishPacket ;
          mqtt:hasPublishVariableHeader ?variable .
  ?variable mqtt:hasTopicName ?topic .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.10u - Which QoS levels are set (in PUBLISH Fixed Header or SubscriptionEntry)?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?entity ?qos WHERE {
  { ?entity a mqtt:PublishFixedHeader ; 
        mqtt:hasQoSLevel ?qos . }
  UNION
  { ?entity a mqtt:SubscriptionEntry ; 
        mqtt:hasQoSLevel ?qos . }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.11u - Are RETAIN/DUP flags set in PUBLISH?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?packet ?retain ?dup WHERE {
  ?packet a mqtt:PublishPacket ;
          mqtt:hasPublishFixedHeader ?header .
  OPTIONAL { ?header mqtt:hasRetainFlag ?retain }
  OPTIONAL { ?header mqtt:hasDUPFlag ?dup }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.12u - Which MQTT Control Packets have which PacketIdentifier?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?packet  ?type ?id WHERE {
  ?packet a mqtt:ControlPacket ;
          a ?type ;
          mqtt:hasVariableHeader ?variable .
  ?variable mqtt:hasPacketIdentifier ?id .

  ?type rdfs:subClassOf* mqtt:ControlPacket .

  FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
  FILTER( ?type != mqtt:ControlPacket )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.13u - Which MQTT SUBSCRIBE payloads contain which SubscriptionEntries?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?payload ?entry WHERE {
  ?payload a mqtt:SubscribePayload ;
           mqtt:hasSubscriptions ?entry .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.14u - Which MQTT Topic Filters (with QoS) are requested by a SubscriptionEntry?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?entry ?filter ?qos WHERE {
  ?entry a mqtt:SubscriptionEntry ;
    OPTIONAL { ?entry mqtt:hasTopicFilter ?filter }  
    OPTIONAL { ?entry mqtt:hasQoSLevel ?qos }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ1.15u - Which MQTT UNSUBSCRIBE payloads remove which MQTT TopicFilters?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?payload ?filter WHERE {
  ?payload a mqtt:UnsubscribePayload ;
           mqtt:hasTopicFilter ?filter .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short